In [1]:
from d3rlpy.algos import BCQ, BCQConfig
from d3rlpy.models.encoders import VectorEncoderFactory

vae_encoder = VectorEncoderFactory([750, 750])
rl_encoder = VectorEncoderFactory([400, 300])
on_server = True


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer
import argparse
import d3rlpy
import gymnasium as gym


dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)
if on_server:
    prefix = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/master_thesis/reproducing_decision_transformer/gymnasium/data/"
else:
    prefix = "/home/julian/programming/cloned_repos/master_thesis/reproducing_decision_transformer/gymnasium/data/"
pkl_path = f"{prefix}{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")


from sklearn.model_selection import train_test_split

algo_train_episodes,fqe_train_episodes = train_test_split(episodes,test_size=0.3,random_state=42)

print(f"Splittet into {len(algo_train_episodes)} Algo Train and {len(fqe_train_episodes)} pisodes.")

buffer_impl = FIFOBuffer(limit=10000000)
algo_train_replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=algo_train_episodes)



args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = "cuda:0" if on_server else "cpu"
args.compile = False


env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3800
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

Hopper-v4
Loaded 3213 episodes.
Splittet into 2249 Algo Train and 964 pisodes.
2025-07-20 20:10.29 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-20 20:10.29 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-20 20:10.29 [info     ] Action size has been automatically determined. action_size=3


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


In [3]:
from d3rlpy.logging import UnifiedFileAdapterFactory

bcq = BCQConfig(
        actor_encoder_factory=rl_encoder,
        actor_learning_rate=1e-3,
        critic_encoder_factory=rl_encoder,
        critic_learning_rate=1e-3,
        imitator_encoder_factory=vae_encoder,
        imitator_learning_rate=1e-3,
        batch_size=100,
        lam=0.75,
        action_flexibility=0.05,
        n_action_samples=100,
        compile_graph=args.compile,
    ).create(args.gpu)

bcq.fit(
    algo_train_replay_buffer,
    n_steps=500,#500000
    n_steps_per_epoch=100,#1000
    save_interval=10,
    evaluators={"environment": d3rlpy.metrics.EnvironmentEvaluator(env)},
    experiment_name=f"BCQ_{args.dataset}_{args.seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    )

2025-07-20 20:11.45 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-20 20:11.45 [debug    ] Building models...            
2025-07-20 20:11.46 [debug    ] Models have been built.       
2025-07-20 20:11.46 [info     ] Directory is created at d3rlpy_logs/BCQ_hopper-medium-expert-v2_1_20250720201146
2025-07-20 20:11.46 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'bcq', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'actor_learning_rate': 0.001, 'critic_learn

Epoch 1/5: 100%|██████████| 100/100 [00:02<00:00, 40.20it/s, vae_loss=0.298, critic_loss=1.12, actor_loss=-4.05]


2025-07-20 20:11.49 [info     ] BCQ_hopper-medium-expert-v2_1_20250720201146: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.002765967845916748, 'time_algorithm_update': 0.02198416233062744, 'vae_loss': 0.29272556319832804, 'critic_loss': 1.022493809722364, 'actor_loss': -4.128766025900841, 'time_step': 0.02481020212173462, 'environment': 114.01103218327808} step=100


Epoch 2/5: 100%|██████████| 100/100 [00:01<00:00, 70.84it/s, vae_loss=0.217, critic_loss=0.0585, actor_loss=-5.66]


2025-07-20 20:11.52 [info     ] BCQ_hopper-medium-expert-v2_1_20250720201146: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.0026885581016540525, 'time_algorithm_update': 0.011308941841125488, 'vae_loss': 0.2164769648015499, 'critic_loss': 0.06487490981817245, 'actor_loss': -5.71983980178833, 'time_step': 0.014059782028198242, 'environment': 307.0001574174075} step=200


...

Epoch 4/5: 100%|██████████| 100/100 [00:01<00:00, 71.03it/s, vae_loss=0.184, critic_loss=0.0821, actor_loss=-8.43]


2025-07-20 20:11.57 [info     ] BCQ_hopper-medium-expert-v2_1_20250720201146: epoch=4 step=400 epoch=4 metrics={'time_sample_batch': 0.0026563549041748046, 'time_algorithm_update': 0.011300089359283448, 'vae_loss': 0.18222981706261634, 'critic_loss': 0.08872510261833667, 'actor_loss': -8.48475634098053, 'time_step': 0.014017508029937745, 'environment': 405.4738714362725} step=400


Epoch 5/5: 100%|██████████| 100/100 [00:01<00:00, 70.65it/s, vae_loss=0.173, critic_loss=0.181, actor_loss=-9.86]


2025-07-20 20:12.02 [info     ] BCQ_hopper-medium-expert-v2_1_20250720201146: epoch=5 step=500 epoch=5 metrics={'time_sample_batch': 0.0026914167404174807, 'time_algorithm_update': 0.011346645355224609, 'vae_loss': 0.17304842486977579, 'critic_loss': 0.17356635577976703, 'actor_loss': -9.916975536346435, 'time_step': 0.014096417427062989, 'environment': 882.7101969544404} step=500


[(1,
  {'time_sample_batch': 0.002765967845916748,
   'time_algorithm_update': 0.02198416233062744,
   'vae_loss': 0.29272556319832804,
   'critic_loss': 1.022493809722364,
   'actor_loss': -4.128766025900841,
   'time_step': 0.02481020212173462,
   'environment': 114.01103218327808}),
 (2,
  {'time_sample_batch': 0.0026885581016540525,
   'time_algorithm_update': 0.011308941841125488,
   'vae_loss': 0.2164769648015499,
   'critic_loss': 0.06487490981817245,
   'actor_loss': -5.71983980178833,
   'time_step': 0.014059782028198242,
   'environment': 307.0001574174075}),
 (3,
  {'time_sample_batch': 0.002697412967681885,
   'time_algorithm_update': 0.011417357921600342,
   'vae_loss': 0.19934502631425857,
   'critic_loss': 0.11320345079526305,
   'actor_loss': -7.066111602783203,
   'time_step': 0.014175343513488769,
   'environment': 239.79928671363095}),
 (4,
  {'time_sample_batch': 0.0026563549041748046,
   'time_algorithm_update': 0.011300089359283448,
   'vae_loss': 0.18222981706261

In [ ]:
bcq = BCQConfig(
        actor_encoder_factory=rl_encoder,
        actor_learning_rate=1e-3,
        critic_encoder_factory=rl_encoder,
        critic_learning_rate=1e-3,
        imitator_encoder_factory=vae_encoder,
        imitator_learning_rate=1e-3,
        batch_size=100,
        lam=0.75,
        action_flexibility=0.05,
        n_action_samples=100,
        compile_graph=args.compile,
    ).create(args.gpu)

bcq.fit(
    algo_train_replay_buffer,
    n_steps=500000,#500000
    n_steps_per_epoch=1000,#1000
    save_interval=100,
    evaluators={"environment": d3rlpy.metrics.EnvironmentEvaluator(env)},
    experiment_name=f"BCQ_{args.dataset}_{args.seed}",
    )

2025-07-18 13:05.59 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-18 13:05.59 [debug    ] Building models...            
2025-07-18 13:05.59 [debug    ] Models have been built.       
2025-07-18 13:05.59 [info     ] Directory is created at d3rlpy_logs/BCQ_hopper-medium-expert-v2_1_20250718130559
2025-07-18 13:05.59 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'bcq', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'actor_learning_rate': 0.001, 'critic_learn

Epoch 1/500: 100%|██████████| 1000/1000 [00:18<00:00, 53.89it/s, vae_loss=0.187, critic_loss=0.34, actor_loss=-10.6]


2025-07-18 13:06.20 [info     ] BCQ_hopper-medium-expert-v2_1_20250718130559: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.004670777797698975, 'time_algorithm_update': 0.013616988182067871, 'vae_loss': 0.186704373896122, 'critic_loss': 0.3405267778784037, 'actor_loss': -10.696544635355473, 'time_step': 0.018405243158340454, 'environment': 744.0440631194859} step=1000


Epoch 2/500: 100%|██████████| 1000/1000 [00:17<00:00, 57.81it/s, vae_loss=0.138, critic_loss=0.913, actor_loss=-25.7]


2025-07-18 13:06.39 [info     ] BCQ_hopper-medium-expert-v2_1_20250718130559: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.00424586033821106, 'time_algorithm_update': 0.012805471897125244, 'vae_loss': 0.138314333088696, 'critic_loss': 0.9146687718480826, 'actor_loss': -25.73957431602478, 'time_step': 0.017166611433029173, 'environment': 491.11532148739104} step=2000


...

Epoch 13/500: 100%|██████████| 1000/1000 [00:25<00:00, 39.73it/s, vae_loss=0.0792, critic_loss=9.1, actor_loss=-160]


2025-07-18 13:11.38 [info     ] BCQ_hopper-medium-expert-v2_1_20250718130559: epoch=13 step=13000 epoch=13 metrics={'time_sample_batch': 0.006345279216766357, 'time_algorithm_update': 0.018475018501281738, 'vae_loss': 0.07918009331822395, 'critic_loss': 9.15639319562912, 'actor_loss': -160.26167260742187, 'time_step': 0.024952624797821045, 'environment': 796.6609882070397} step=13000


Epoch 14/500: 100%|██████████| 1000/1000 [00:21<00:00, 46.06it/s, vae_loss=0.0783, critic_loss=10.5, actor_loss=-169]


2025-07-18 13:12.04 [info     ] BCQ_hopper-medium-expert-v2_1_20250718130559: epoch=14 step=14000 epoch=14 metrics={'time_sample_batch': 0.005395142555236816, 'time_algorithm_update': 0.016014375925064085, 'vae_loss': 0.0782238290719688, 'critic_loss': 10.494884705066681, 'actor_loss': -169.18205184936522, 'time_step': 0.021532698392868043, 'environment': 1080.1490646149275} step=14000


Epoch 15/500:  86%|████████▌ | 856/1000 [00:20<00:03, 39.64it/s, vae_loss=0.0768, critic_loss=13.2, actor_loss=-177]